# EDA — Bank Marketing Dataset

Datathon 7MLET — Grupo XX

Base: [Bank Marketing Dataset (Kaggle)](https://www.kaggle.com/datasets/henriqueyamahata/bank-marketing)

Objetivo: entender a distribuição da variável alvo (conversão) e preparar as features do cliente para o modelo de bandit adaptativo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

## 1. Carregar dados

Baixe o `bank-additional-full.csv` (separador `;`) do Kaggle e coloque em `data/bank-additional-full.csv`.

In [ ]:
df = pd.read_csv('../data/bank-additional-full.csv', sep=';')
print(df.shape)
df.head()

In [ ]:
df.info()
df.isnull().sum()

## 2. Variável alvo (conversão)

`y` indica se o cliente aceitou o depósito a prazo (`yes`/`no`). É desbalanceada (~11% de conversão).

In [ ]:
df['target'] = (df['y'] == 'yes').astype(int)
print(df['target'].value_counts(normalize=True))

plt.figure(figsize=(4,4))
df['target'].value_counts().plot(kind='bar', color=['#3b3b3b', '#2ecc71'])
plt.title('Distribuição da variável alvo (conversão)')
plt.xticks([0,1], ['Não converteu', 'Converteu'], rotation=0)
plt.show()

## 3. Remoção de coluna de vazamento temporal

`duration` só é conhecida DEPOIS da ligação acontecer — ela vaza informação do futuro e não pode ser usada como feature. Removemos.

In [ ]:
df = df.drop(columns=['duration'])
df.shape

## 4. Análise das variáveis categóricas relevantes para o contexto do bandit

Vamos usar um subconjunto de variáveis do cliente como **contexto** para a decisão adaptativa: idade, profissão, estado civil, escolaridade, se tem inadimplência, habitação, empréstimo pessoal, e o mês/dia de contato (proxy de canal/tempo).

In [ ]:
context_cols = ['age', 'job', 'marital', 'education', 'default',
                'housing', 'loan', 'contact', 'month', 'campaign', 'pdays', 'previous']

for col in ['job', 'marital', 'education', 'contact']:
    print(f"--- {col} ---")
    print(df[col].value_counts())
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
sns.histplot(df['age'], bins=30, ax=axes[0])
axes[0].set_title('Distribuição de idade')

conv_by_job = df.groupby('job')['target'].mean().sort_values(ascending=False)
conv_by_job.plot(kind='bar', ax=axes[1], color='#2ecc71')
axes[1].set_title('Taxa de conversão por profissão')
plt.tight_layout()
plt.show()

## 5. Tratamento de dados

- One-hot encoding das variáveis categóricas de contexto.
- Padronização da idade e variáveis numéricas.
- Salvamos um dataset limpo pronto para o notebook do bandit (Etapa 2 e 3).

In [ ]:
from sklearn.preprocessing import StandardScaler

df_model = df[context_cols + ['target']].copy()

# 'pdays' = 999 significa "nunca contatado antes" -> vira flag binária
df_model['nunca_contatado'] = (df_model['pdays'] == 999).astype(int)
df_model['pdays'] = df_model['pdays'].replace(999, -1)

cat_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month']
df_encoded = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

num_cols = ['age', 'campaign', 'pdays', 'previous']
scaler = StandardScaler()
df_encoded[num_cols] = scaler.fit_transform(df_encoded[num_cols])

print(df_encoded.shape)
df_encoded.head()

In [ ]:
df_encoded.to_csv('../data/bank_marketing_clean.csv', index=False)
print("Dataset limpo salvo em data/bank_marketing_clean.csv")

## 6. Simulação de braços (ofertas)

O dataset original tem apenas 1 "oferta" (depósito a prazo) e seu resultado. Para simular o cenário de **múltiplas ofertas/mensagens** do desafio, criamos 3 braços fictícios (Oferta A, B, C) aplicando pequenas variações de probabilidade sobre a propensão real do cliente — isso simula canais/mensagens diferentes sem inventar dados de cliente novos.

Isso é usado na Etapa 2/3 (notebook `02_bandit_mlflow.ipynb`).

In [ ]:
np.random.seed(42)

# fator multiplicativo de resposta por braço (assunção documentada e simples)
arm_factors = {'Oferta_A': 1.0, 'Oferta_B': 0.85, 'Oferta_C': 1.15}

df_arms = df_encoded.copy()
base_prob = df_arms['target'].rolling(window=50, min_periods=1).mean().fillna(df_arms['target'].mean())

for arm, factor in arm_factors.items():
    prob = np.clip(base_prob * factor, 0, 1)
    df_arms[f'reward_{arm}'] = np.random.binomial(1, prob)

df_arms.to_csv('../data/bank_marketing_arms.csv', index=False)
df_arms[[c for c in df_arms.columns if 'reward' in c]].mean()